# Part 4 - Metin Gömme (Text Embedding) İşlemleri

Projenin bu aşamasında, ilk adımda elde edilen 100 orijinal soru ve Claude API ile üretilen 2500 sentetik (augmented) soru birleştirilerek tek bir veri seti haline getirilmiştir.

Toplam 2600 soruluk bu genişletilmiş veri setindeki metinler, makine öğrenmesi modelleri tarafından anlaşılabilecek sayısal vektörlere dönüştürülmüştür. Bu işlem için Türkçe dilinde yüksek performans gösteren `ytu-ce-cosmos/turkish-e5-large` sentence-transformer modeli kullanılmıştır. Üretilen vektörler (embeddings), ilerleyen aşamalarda semantik arama, kümeleme (clustering) veya boyut azaltma (PCA/t-SNE) işlemleri için `.npy` formatında kaydedilmektedir.

## 1. Kurulum ve Ortam Hazırlığı
Metinleri vektörlere dönüştürmek için `sentence-transformers` kütüphanesini kuruyor ve verilerimizi kaydetmek için Google Drive bağlantımızı yapıyoruz.

In [ ]:
# Gerekli kütüphanelerin kurulumu
!pip install -q sentence-transformers

from google.colab import drive
drive.mount('/content/drive')

## 2. Embedding Modelinin Yüklenmesi
Hugging Face üzerinden Cosmos takımının Türkçe için özel olarak eğittiği `turkish-e5-large` modelini içe aktarıyoruz. Bu model, cümleleri anlamsal (semantik) olarak 1024 boyutlu (veya model tipine göre değişken) vektör uzayında ifade etmemizi sağlar.

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import os

# Türkçe için optimize edilmiş E5 large modelini yüklüyoruz
EMBEDDING_MODEL = "ytu-ce-cosmos/turkish-e5-large"
emb_model = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding modeli başarıyla yüklendi.")

## 3. Orijinal Verilerin Hazırlanması
Birinci aşamada elde ettiğimiz 50 başarılı ve 50 başarısız orijinal Qwen 3.5 4B yanıtlarını yüklüyoruz. Modellerin daha sonraki analizlerde ayırt edilebilmesi için `soru_turu` ve `strategy` gibi etiketleri ekliyoruz. Başarılı dosyadan gelen veriler için `qwen_correct` sütununu `True`, başarısızlar için `False` olarak işaretliyoruz.

In [ ]:
import re

# Dosya yolları
RESULTS_PATH          = "/content/drive/MyDrive/kaynaklar/augmented_dataset_with_results.csv"
ORIGINAL_PATH_SUCCESS = "/content/drive/MyDrive/kaynaklar/qwen3_5_4b_basarili_50.csv"
ORIGINAL_PATH_FAIL    = "/content/drive/MyDrive/kaynaklar/qwen3_5_4b_basarisiz_50.csv"

# Orijinal 100 sorunun yüklenmesi
df_s = pd.read_csv(ORIGINAL_PATH_SUCCESS)
df_f = pd.read_csv(ORIGINAL_PATH_FAIL)

# Durumların etiketlenmesi
df_s['original_status'] = 'basarili'
df_f['original_status'] = 'basarisiz'

# Verilerin birleştirilmesi
df_orig = pd.concat([df_s, df_f], ignore_index=True)

# Başarılı dosyadaki sorular zaten doğru çözülmüş, başarısızlar yanlış
df_orig['qwen_correct'] = df_orig['original_status'].map({
    'basarili':  True,
    'basarisiz': False
})

# Analiz metadatalarının eklenmesi
df_orig['soru_turu'] = 'orijinal'
df_orig['strategy']  = 'Orijinal'
df_orig['metin']     = df_orig['question']

print(f"Orijinal soru sayısı: {len(df_orig)}")
print("Qwen Doğruluk Dağılımı (Orijinal):")
print(df_orig['qwen_correct'].value_counts())

## 4. Üretilen Verilerle Birleştirme
Üçüncü aşamada modelden geçirilmiş olan 2500 soruluk genişletilmiş (augmented) veri setini yüklüyor ve orijinal 100 soru ile tek bir DataFrame altında birleştiriyoruz. Toplamda 2600 satırlık nihai bir havuz elde ediyoruz.

In [ ]:
# Üretilen 2500 soruyu yükle ve etiketle
df_aug = pd.read_csv(RESULTS_PATH)
df_aug['soru_turu'] = 'uretilen'
df_aug['metin']     = df_aug['generated_question']

# Sadece gerekli sütunları alarak orijinal ve üretilen verileri birleştir
df_all = pd.concat(
    [df_orig[['metin', 'soru_turu', 'strategy', 'original_status', 'qwen_correct']],
     df_aug[['metin', 'soru_turu', 'strategy', 'original_status', 'qwen_correct']]],
    ignore_index=True
)

print(f"\nToplam işlenecek soru: {len(df_all)}")
print("\nSoru Türü Dağılımı:")
print(df_all['soru_turu'].value_counts())
print(f"\nGenel qwen_correct (Doğruluk) Dağılımı:")
print(df_all['qwen_correct'].value_counts())

## 5. Vektör Dönüşümü (Embedding Hesaplama)
Birleştirilen 2600 metin için vektör (embedding) hesabı yapılır. Her hesaplama baştan yapılmasın diye, dosyanın `Drive` üzerinde var olup olmadığı kontrol edilir (Checkpoint mekanizması). Yeni hesaplama yapılacaksa `batch_size=64` kullanılarak işlemler hızlandırılır ve hesaplanan vektörler `.npy` (NumPy array) formatında dışa aktarılır.

In [ ]:
EMBEDDINGS_PATH = "/content/drive/MyDrive/kaynaklar/embeddings.npy"
METADATA_PATH   = "/content/drive/MyDrive/kaynaklar/embeddings_metadata.csv"

# Eğer embedding matrisi zaten hesaplanıp kaydedildiyse tekrar hesaplama (zaman tasarrufu)
if os.path.exists(EMBEDDINGS_PATH):
    embeddings = np.load(EMBEDDINGS_PATH)
    df_meta    = pd.read_csv(METADATA_PATH)
    print(f"Mevcut embedding dosyası yüklendi. Vektör boyutu: {embeddings.shape}")
else:
    print("Embedding hesaplanıyor... Bu işlem GPU hızına bağlı olarak ~5-10 dakika sürebilir.")

    # NaN değerleri olası hatalara karşı temizle
    metinler = df_all['metin'].fillna('').tolist()

    # Vektörleri hesapla (Cosine similarity işlemlerini kolaylaştırmak için normalize ediyoruz)
    embeddings = emb_model.encode(
        metinler,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    # Vektörleri ve verisetini metadata olarak kaydet
    np.save(EMBEDDINGS_PATH, embeddings)
    df_all.to_csv(METADATA_PATH, index=False)

    print(f"Vektörler başarıyla kaydedildi. Vektör boyutu: {embeddings.shape}")
    print(f"Metadata (etiketler) kaydedildi: {METADATA_PATH}")